<a href="https://colab.research.google.com/github/MeenakshiRajpurohit/CMPE-255-Data-Mining/blob/main/MovieLensDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
from google.colab import files
uploaded = files.upload()   # picks file from your Mac

Saving xkgrank_save.zip to xkgrank_save.zip


In [4]:
import zipfile
with zipfile.ZipFile("/content/xkgrank_save.zip") as zf:
    zf.extractall("/content/")

import os
DATA = "/content/xkgrank_save/"
print(f"Files in {DATA}:")
for f in sorted(os.listdir(DATA)):
    print(f"   {f}")

Files in /content/xkgrank_save/:
   all_movie_ids.json
   all_user_ids.json
   df_movies.csv
   df_ratings.csv
   df_test.csv
   df_train.csv
   df_users.csv
   df_valid.csv
   edge_index.pt
   item2idx.json
   kg_graph.pkl
   sizes.json
   user2idx.json


In [5]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:    {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nUsing: {device}")

CUDA available: True
GPU:    NVIDIA H100 80GB HBM3
Memory: 85.0 GB

Using: cuda


In [6]:
!pip install torch-geometric -q
print("✅ torch-geometric installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 93.9 MB/s eta 0:00:00
✅ torch-geometric installed


In [7]:
import pandas as pd
import json
import pickle
from collections import defaultdict

# DataFrames
df_train  = pd.read_csv(f"{DATA}df_train.csv")
df_valid  = pd.read_csv(f"{DATA}df_valid.csv")
df_test   = pd.read_csv(f"{DATA}df_test.csv")
df_movies = pd.read_csv(f"{DATA}df_movies.csv")

# Mappings — convert string keys back to int
with open(f"{DATA}user2idx.json") as f:
    user2idx = {int(k): v for k, v in json.load(f).items()}
with open(f"{DATA}item2idx.json") as f:
    item2idx = {int(k): v for k, v in json.load(f).items()}
with open(f"{DATA}all_user_ids.json") as f:
    all_user_ids = json.load(f)
with open(f"{DATA}all_movie_ids.json") as f:
    all_movie_ids = json.load(f)
with open(f"{DATA}sizes.json") as f:
    sizes = json.load(f)

n_users   = sizes['n_users']
n_items   = sizes['n_items']
num_nodes = sizes['num_nodes']

# edge_index → GPU
edge_index = torch.load(f"{DATA}edge_index.pt").to(device)

# NetworkX graph
with open(f"{DATA}kg_graph.pkl", 'rb') as f:
    G = pickle.load(f)

print("✅ Everything reloaded")
print(f"   Train: {len(df_train):,} | Valid: {len(df_valid):,} | Test: {len(df_test):,}")
print(f"   Users: {n_users:,} | Movies: {n_items:,} | Total nodes: {num_nodes:,}")
print(f"   edge_index: {edge_index.shape} on {edge_index.device}")
print(f"   Graph: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")

✅ Everything reloaded
   Train: 988,129 | Valid: 6,040 | Test: 6,040
   Users: 6,040 | Movies: 3,704 | Total nodes: 9,744
   edge_index: torch.Size([2, 1976258]) on cuda:0
   Graph: 9,762 nodes, 999,264 edges


In [8]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import LGConv

class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, num_layers=3):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        self.convs     = nn.ModuleList([LGConv() for _ in range(num_layers)])
        nn.init.xavier_uniform_(self.embedding.weight)

    def forward(self, edge_index):
        x = self.embedding.weight
        embs = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            embs.append(x)
        return F.normalize(torch.stack(embs, dim=1).mean(dim=1), p=2, dim=-1)

    def bpr_loss(self, u, p, n, reg=1e-4):
        pos = (u * p).sum(dim=1)
        neg = (u * n).sum(dim=1)
        bpr = -F.logsigmoid(pos - neg).mean()
        return bpr + reg * (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / u.shape[0]

# Tuned hyperparameters for MovieLens-1M
EMBEDDING_DIM = 64
NUM_LAYERS    = 3
EPOCHS        = 100
BATCH_SIZE    = 4096
PATIENCE      = 8

model     = LightGCN(num_nodes, EMBEDDING_DIM, NUM_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)

print(f"✅ Model on {device}")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

✅ Model on cuda
   Parameters: 623,616


In [9]:
import numpy as np
import time

u_pos_items = defaultdict(set)
for _, row in df_train.iterrows():
    u_pos_items[row['user_id']].add(row['movie_id'])

train_pairs = [(row['user_id'], row['movie_id'])
               for _, row in df_train.iterrows()
               if row['user_id'] in user2idx and row['movie_id'] in item2idx]

u_all = torch.tensor([user2idx[u] for u,_ in train_pairs], dtype=torch.long).to(device)
p_all = torch.tensor([item2idx[m] for _,m in train_pairs], dtype=torch.long).to(device)
item_idx_tensor = torch.tensor(list(item2idx.values()), dtype=torch.long).to(device)
n_pairs = len(u_all)

val_gt = defaultdict(set)
for _, row in df_valid.iterrows():
    val_gt[row['user_id']].add(row['movie_id'])

print(f"✅ Training: {n_pairs:,} pairs on {device}")
print(f"   Items: {len(item_idx_tensor):,}")
print(f"   Val users: {len(val_gt):,}")

✅ Training: 988,129 pairs on cuda
   Items: 3,704
   Val users: 6,040


In [11]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import LGConv

class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, num_layers=3, dropout=0.0):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        self.convs     = nn.ModuleList([LGConv() for _ in range(num_layers)])
        self.dropout   = nn.Dropout(dropout)
        nn.init.xavier_uniform_(self.embedding.weight)

    def forward(self, edge_index):
        x = self.dropout(self.embedding.weight)
        embs = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            embs.append(x)
        return F.normalize(torch.stack(embs, dim=1).mean(dim=1), p=2, dim=-1)

    def bpr_loss(self, u, p, n, reg=1e-4):
        pos = (u * p).sum(dim=1)
        neg = (u * n).sum(dim=1)
        bpr = -F.logsigmoid(pos - neg).mean()
        return bpr + reg * (u.norm(2).pow(2) + p.norm(2).pow(2) + n.norm(2).pow(2)) / u.shape[0]

# BIGGER model — H100 can easily handle this
EMBEDDING_DIM = 128       # ↑ from 64
NUM_LAYERS    = 3
EPOCHS        = 200       # ↑ from 100
BATCH_SIZE    = 8192      # ↑ from 4096 (H100 has 80GB!)
PATIENCE      = 15        # ↑ from 8 (less aggressive early stop)
LR            = 0.001
WEIGHT_DECAY  = 1e-5      # ↓ from 1e-4 (less regularization on dense data)

model     = LightGCN(num_nodes, EMBEDDING_DIM, NUM_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

print(f"✅ Model — {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"   Embedding dim: {EMBEDDING_DIM}")
print(f"   Layers:        {NUM_LAYERS}")
print(f"   Max epochs:    {EPOCHS}")
print(f"   Batch size:    {BATCH_SIZE}")
print(f"   LR:            {LR}")
print(f"   Weight decay:  {WEIGHT_DECAY}")

✅ Model — 1,247,232 parameters
   Embedding dim: 128
   Layers:        3
   Max epochs:    200
   Batch size:    8192
   LR:            0.001
   Weight decay:  1e-05


In [16]:
import pandas as pd

df_ratings = pd.read_csv(f"{DATA}df_ratings.csv")
print(f"✅ df_ratings loaded: {df_ratings.shape}")
print(df_ratings.head(3))

✅ df_ratings loaded: (1000209, 4)
   user_id  movie_id  rating  timestamp
0        1      3186       4  978300019
1        1      1270       5  978300055
2        1      1721       4  978300055


In [17]:
import numpy as np
import pandas as pd

np.random.seed(42)
print("Re-splitting with random leave-one-out (standard for MovieLens-1M)...")

# Filter users with enough ratings
user_counts = df_ratings.groupby('user_id').size()
valid_users = user_counts[user_counts >= 10].index
df_full = df_ratings[df_ratings['user_id'].isin(valid_users)].copy()
print(f"Users with ≥10 ratings: {len(valid_users):,}")

# RANDOM leave-one-out (not time-based)
test_idx, valid_idx, train_idx = [], [], []
for user_id, group in df_full.groupby('user_id'):
    indices = group.index.tolist()
    np.random.shuffle(indices)
    test_idx.append(indices[0])
    valid_idx.append(indices[1])
    train_idx.extend(indices[2:])

df_test  = df_full.loc[test_idx].reset_index(drop=True)
df_valid = df_full.loc[valid_idx].reset_index(drop=True)
df_train = df_full.loc[train_idx].reset_index(drop=True)

print(f"\nTrain: {df_train.shape}")
print(f"Valid: {df_valid.shape}")
print(f"Test:  {df_test.shape}")
print(f"Users: {df_train['user_id'].nunique():,}")
print(f"Items: {df_train['movie_id'].nunique():,}")

Re-splitting with random leave-one-out (standard for MovieLens-1M)...
Users with ≥10 ratings: 6,040

Train: (988129, 4)
Valid: (6040, 4)
Test:  (6040, 4)
Users: 6,040
Items: 3,704


In [18]:
import numpy as np
import pandas as pd

np.random.seed(42)
print("Re-splitting with random leave-one-out (standard for MovieLens-1M)...")

# Filter users with enough ratings
user_counts = df_ratings.groupby('user_id').size()
valid_users = user_counts[user_counts >= 10].index
df_full = df_ratings[df_ratings['user_id'].isin(valid_users)].copy()
print(f"Users with ≥10 ratings: {len(valid_users):,}")

# RANDOM leave-one-out (not time-based)
test_idx, valid_idx, train_idx = [], [], []
for user_id, group in df_full.groupby('user_id'):
    indices = group.index.tolist()
    np.random.shuffle(indices)
    test_idx.append(indices[0])
    valid_idx.append(indices[1])
    train_idx.extend(indices[2:])

df_test  = df_full.loc[test_idx].reset_index(drop=True)
df_valid = df_full.loc[valid_idx].reset_index(drop=True)
df_train = df_full.loc[train_idx].reset_index(drop=True)

print(f"\nTrain: {df_train.shape}")
print(f"Valid: {df_valid.shape}")
print(f"Test:  {df_test.shape}")
print(f"Users: {df_train['user_id'].nunique():,}")
print(f"Items: {df_train['movie_id'].nunique():,}")

Re-splitting with random leave-one-out (standard for MovieLens-1M)...
Users with ≥10 ratings: 6,040

Train: (988129, 4)
Valid: (6040, 4)
Test:  (6040, 4)
Users: 6,040
Items: 3,704


In [19]:
from collections import defaultdict
from torch_geometric.utils import to_undirected

# Rebuild mappings
all_user_ids   = sorted(df_train['user_id'].unique())
all_movie_ids  = sorted(df_train['movie_id'].unique())
n_users        = len(all_user_ids)
n_items        = len(all_movie_ids)

user2idx = {uid: i for i, uid in enumerate(all_user_ids)}
item2idx = {mid: i + n_users for i, mid in enumerate(all_movie_ids)}
num_nodes = n_users + n_items

# Rebuild edge_index
edge_src, edge_tgt = [], []
for _, row in df_train.iterrows():
    if row['user_id'] in user2idx and row['movie_id'] in item2idx:
        u = user2idx[row['user_id']]
        i = item2idx[row['movie_id']]
        edge_src += [u, i]
        edge_tgt += [i, u]
edge_index = torch.tensor([edge_src, edge_tgt], dtype=torch.long).to(device)

# Rebuild training pairs
u_pos_items = defaultdict(set)
for _, row in df_train.iterrows():
    u_pos_items[row['user_id']].add(row['movie_id'])

train_pairs = [(row['user_id'], row['movie_id'])
               for _, row in df_train.iterrows()
               if row['user_id'] in user2idx and row['movie_id'] in item2idx]
u_all = torch.tensor([user2idx[u] for u,_ in train_pairs], dtype=torch.long).to(device)
p_all = torch.tensor([item2idx[m] for _,m in train_pairs], dtype=torch.long).to(device)
item_idx_tensor = torch.tensor(list(item2idx.values()), dtype=torch.long).to(device)
n_pairs = len(u_all)

# Rebuild val ground truth
val_gt = defaultdict(set)
for _, row in df_valid.iterrows():
    val_gt[row['user_id']].add(row['movie_id'])

# Rebuild eval tensors
movie_id_to_pos = {mid: i for i, mid in enumerate(all_movie_ids)}
n_movies = len(all_movie_ids)

val_users_list = [u for u in val_gt.keys() if u in user2idx]
val_user_idx_t = torch.tensor([user2idx[u] for u in val_users_list], dtype=torch.long, device=device)

seen_mask_val = torch.zeros((len(val_users_list), n_movies), dtype=torch.bool, device=device)
for i, uid in enumerate(val_users_list):
    for mid in u_pos_items[uid]:
        if mid in movie_id_to_pos:
            seen_mask_val[i, movie_id_to_pos[mid]] = True

val_gt_idx = []
for uid in val_users_list:
    gt_mids = [movie_id_to_pos[m] for m in val_gt[uid] if m in movie_id_to_pos]
    val_gt_idx.append(set(gt_mids))

print(f"✅ Rebuilt — {n_pairs:,} train pairs, {len(val_users_list):,} val users")
print(f"   edge_index: {edge_index.shape}")

✅ Rebuilt — 988,129 train pairs, 6,040 val users
   edge_index: torch.Size([2, 1976258])


In [21]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import LGConv
import numpy as np
import time

class LightGCN(nn.Module):
    def __init__(self, num_nodes, embedding_dim=64, num_layers=3):
        super().__init__()
        self.embedding = nn.Embedding(num_nodes, embedding_dim)
        self.convs = nn.ModuleList([LGConv() for _ in range(num_layers)])
        nn.init.normal_(self.embedding.weight, std=0.1)   # standard init for LightGCN

    def forward(self, edge_index):
        x = self.embedding.weight
        embs = [x]
        for conv in self.convs:
            x = conv(x, edge_index)
            embs.append(x)
        return torch.stack(embs, dim=1).mean(dim=1)


# ── REBUILD edge_index — UNDIRECTED only (don't manually add reverse) ─────
print("Rebuilding edge_index (single direction)...")
edge_src, edge_tgt = [], []
for _, row in df_train.iterrows():
    if row['user_id'] in user2idx and row['movie_id'] in item2idx:
        edge_src.append(user2idx[row['user_id']])
        edge_tgt.append(item2idx[row['movie_id']])

# Make undirected via PyG utility (correct way)
from torch_geometric.utils import to_undirected
edge_index_dir = torch.tensor([edge_src, edge_tgt], dtype=torch.long)
edge_index = to_undirected(edge_index_dir).to(device)
print(f"✅ edge_index: {edge_index.shape}")


# ── Hyperparameters (from published LightGCN paper) ──
EMBEDDING_DIM = 64
NUM_LAYERS    = 3
EPOCHS        = 200
BATCH_SIZE    = 2048
PATIENCE      = 30
LR            = 0.001
L2_REG        = 1e-4

model = LightGCN(num_nodes, EMBEDDING_DIM, NUM_LAYERS).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)


# ── TRAIN — proper per-batch forward/backward ──
best_ndcg, patience_ctr, best_weights = 0.0, 0, None
train_losses, val_ndcgs, val_recalls, val_epochs = [], [], [], []

print(f"\nTraining canonical LightGCN — {n_pairs:,} pairs\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    t0 = time.time()

    # Shuffle once per epoch
    perm = torch.randperm(n_pairs, device=device)
    u_shuf = u_all[perm]
    p_shuf = p_all[perm]
    n_shuf = item_idx_tensor[torch.randint(0, len(item_idx_tensor), (n_pairs,), device=device)]

    total_loss = 0.0
    n_batches = 0

    for start in range(0, n_pairs, BATCH_SIZE):
        u_b = u_shuf[start:start+BATCH_SIZE]
        p_b = p_shuf[start:start+BATCH_SIZE]
        n_b = n_shuf[start:start+BATCH_SIZE]

        # ✅ Forward INSIDE the batch loop
        optimizer.zero_grad()
        embs = model(edge_index)

        u_e = embs[u_b]
        p_e = embs[p_b]
        n_e = embs[n_b]

        # BPR loss
        pos_scores = (u_e * p_e).sum(dim=1)
        neg_scores = (u_e * n_e).sum(dim=1)
        bpr_loss = -F.logsigmoid(pos_scores - neg_scores).mean()

        # L2 reg on raw embedding lookup (not propagated)
        raw_u = model.embedding(u_b)
        raw_p = model.embedding(p_b)
        raw_n = model.embedding(n_b)
        reg_loss = L2_REG * (raw_u.norm(2).pow(2) + raw_p.norm(2).pow(2) + raw_n.norm(2).pow(2)) / (2 * len(u_b))

        loss = bpr_loss + reg_loss
        loss.backward()       # ✅ NO retain_graph
        optimizer.step()

        total_loss += loss.item()
        n_batches  += 1

    avg_loss = total_loss / n_batches
    train_losses.append(avg_loss)

    # ── Eval every 5 epochs ──
    if epoch % 5 == 0:
        model.eval()
        with torch.no_grad():
            embs_eval = model(edge_index)
            item_embs = embs_eval[item_idx_tensor]
            user_embs = embs_eval[val_user_idx_t]
            scores    = user_embs @ item_embs.T
            scores[seen_mask_val] = -9999.0
            top10_idx = scores.topk(10, dim=1).indices.cpu().numpy()

        val_ndcg_list, val_recall_list = [], []
        for i, gt_set in enumerate(val_gt_idx):
            if not gt_set: continue
            recs = top10_idx[i]
            hits = sum(1 for r in recs if int(r) in gt_set)
            dcg  = sum(1/np.log2(r+2) for r,it in enumerate(recs) if int(it) in gt_set)
            idcg = sum(1/np.log2(r+2) for r in range(min(len(gt_set), 10)))
            val_ndcg_list.append(dcg/idcg if idcg > 0 else 0.0)
            val_recall_list.append(hits / len(gt_set))

        val_ndcg = np.mean(val_ndcg_list)
        val_recall = np.mean(val_recall_list)
        val_ndcgs.append(val_ndcg)
        val_recalls.append(val_recall)
        val_epochs.append(epoch)

        if val_ndcg > best_ndcg:
            best_ndcg = val_ndcg
            best_weights = {k: v.clone() for k,v in model.state_dict().items()}
            patience_ctr = 0
            flag = "✅ BEST"
        else:
            patience_ctr += 1
            flag = f"patience {patience_ctr}/{PATIENCE}"

        print(f"Epoch {epoch:>3}/{EPOCHS} | Loss: {avg_loss:.4f} | "
              f"NDCG@10: {val_ndcg:.4f} | Recall@10: {val_recall:.4f} | "
              f"{time.time()-t0:.1f}s | {flag}")

        if patience_ctr >= PATIENCE:
            print(f"\n⏹ Early stop — best NDCG@10: {best_ndcg:.4f}")
            break

if best_weights:
    model.load_state_dict(best_weights)
model.eval()
with torch.no_grad():
    final_embeddings = model(edge_index)

print(f"\n✅ Training complete — best Val NDCG@10: {best_ndcg:.4f}")

Rebuilding edge_index (single direction)...
✅ edge_index: torch.Size([2, 1976258])

Training canonical LightGCN — 988,129 pairs

Epoch   5/200 | Loss: 0.3740 | NDCG@10: 0.0439 | Recall@10: 0.0868 | 7.2s | ✅ BEST
Epoch  10/200 | Loss: 0.3273 | NDCG@10: 0.0669 | Recall@10: 0.1205 | 7.2s | ✅ BEST
Epoch  15/200 | Loss: 0.3019 | NDCG@10: 0.0739 | Recall@10: 0.1305 | 7.2s | ✅ BEST
Epoch  20/200 | Loss: 0.2879 | NDCG@10: 0.0786 | Recall@10: 0.1436 | 7.2s | ✅ BEST
Epoch  25/200 | Loss: 0.2772 | NDCG@10: 0.0840 | Recall@10: 0.1533 | 7.2s | ✅ BEST
Epoch  30/200 | Loss: 0.2701 | NDCG@10: 0.0883 | Recall@10: 0.1623 | 7.2s | ✅ BEST
Epoch  35/200 | Loss: 0.2643 | NDCG@10: 0.0894 | Recall@10: 0.1646 | 7.2s | ✅ BEST
Epoch  40/200 | Loss: 0.2586 | NDCG@10: 0.0917 | Recall@10: 0.1687 | 7.2s | ✅ BEST
Epoch  45/200 | Loss: 0.2551 | NDCG@10: 0.0943 | Recall@10: 0.1734 | 7.2s | ✅ BEST
Epoch  50/200 | Loss: 0.2512 | NDCG@10: 0.0953 | Recall@10: 0.1739 | 7.2s | ✅ BEST
Epoch  55/200 | Loss: 0.2478 | NDCG@10: 0